# Kaggle 01 — COBOL Corpus Ingest

**Week 2 task** — Run on Kaggle CPU (32 cores, niente GPU necessaria).

Ingest pipeline (priorità):
1. XMainframe Training Corpus (236M token, MIT) — `Fsoft-AIC/XMainframe`
2. The Stack v2 dedup — filter `lang=cobol`
3. X-COBOL Zenodo (85 repo, ~3.6M token) — dataset Kaggle `xcobol`
4. Community repos — clonati inline
5. NIST COBOL 85 test suite — da gnucobol-bin

Output: HF Hub private dataset `AlexThunder0/cobol-cpt-corpus`

**Prerequisiti Kaggle Secrets:** `HF_TOKEN`, `GH_TOKEN`

In [ ]:
!pip install datasets huggingface-hub datasketch tqdm -q

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
GH_TOKEN = secrets.get_secret('GH_TOKEN')

print('Secrets configurati')

In [ ]:
!sudo apt-get install -y gnucobol -q
!cobc --version

In [ ]:
import subprocess, sys, os

# Scrivi .netrc per autenticazione GitHub — più affidabile dell'URL con token
with open(os.path.expanduser('~/.netrc'), 'w') as f:
    f.write(f'machine github.com login x-access-token password {GH_TOKEN}\n')
os.chmod(os.path.expanduser('~/.netrc'), 0o600)

subprocess.run(
    ['git', 'clone', '--depth', '1', '--quiet',
     'https://github.com/AlexThunder01/Qwen-COBOL.git',
     '/kaggle/working/Qwen-COBOL'],
    check=True,
    env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'},
)
sys.path.insert(0, '/kaggle/working/Qwen-COBOL')
print('Repo clonato')

In [ ]:
import os
os.makedirs('/kaggle/working/community_repos', exist_ok=True)

repos = [
    'https://github.com/openmainframeproject/cobol-programming-course',
    'https://github.com/Martinfx/COBOL',
]
for url in repos:
    name = url.split('/')[-1]
    dest = f'/kaggle/working/community_repos/{name}'
    if not os.path.exists(dest):
        ret = os.system(f'git clone --depth 1 --quiet {url} {dest}')
        if ret == 0:
            print(f'Clonato: {name}')
        else:
            print(f'ERRORE clone: {name}')
    else:
        print(f'Già presente: {name}')

In [ ]:
from src.pipeline.ingest import run_ingest
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

run_ingest(
    zenodo_dir='/kaggle/input/datasets/alexander912/xcobol',
    repos_dir='/kaggle/working/community_repos',
    gnucobol_share='/usr/share/gnucobol',
    skip_stack=False,
    skip_xmainframe=False,
)

In [ ]:
from datasets import load_dataset
import os

ds = load_dataset(
    'AlexThunder0/cobol-cpt-corpus',
    split='train',
    streaming=True,
    token=os.environ['HF_TOKEN'],
)
total_chars = 0
n = 0
for row in ds:
    total_chars += len(row['content'])
    n += 1
    if n % 10000 == 0:
        print(f'{n:,} records — ~{total_chars/4:,.0f} token (stima)')

print(f'\nTotale: {n:,} records — ~{total_chars/4:,.0f} token')
print(f'Target: 100-300M token')